# Flux2LoraTrainer vs Flux2Pipeline Comparison

This notebook compares the T2I (Text-to-Image) outputs of `Flux2LoraTrainer` with the original `Flux2Pipeline`.

For I2I (Image-to-Image) comparison, run: `python tests/src/trainer/compare_flux2_i2i.py`

In [ ]:
# Setup
import torch
import numpy as np
from dotenv import load_dotenv
import sys
import os

# Load environment
load_dotenv()

# Add src to path
sys.path.insert(0, os.path.abspath("../../../src"))

from qflux.trainer.flux2_trainer import Flux2LoraTrainer
from qflux.data.config import load_config_from_yaml
from diffusers import Flux2Pipeline

device = "cuda"
model_path = "diffusers/FLUX.2-dev-bnb-4bit"

In [ ]:
# Load original Flux2Pipeline
print("Loading Flux2Pipeline...")
pipe = Flux2Pipeline.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
)
pipe.to(device)
print("Pipeline loaded")

In [ ]:
# Load Flux2LoraTrainer
print("Loading Flux2LoraTrainer...")
config_path = "../../../configs/flux2_t2i_fp4_config.yaml"
config = load_config_from_yaml(config_path)
trainer_t2i = Flux2LoraTrainer(config)
trainer_t2i.setup_predict()
print("Trainer loaded")

In [ ]:
# Test prompt
prompt = "A beautiful portrait of a woman with flowing dark blue hair"
height = 832
width = 576
num_inference_steps = 20
guidance_scale = 3.5
seed = 42

In [ ]:
# ============================================================
# 1. ENCODE PROMPT COMPARISON
# ============================================================
print("=" * 60)
print("1. ENCODE PROMPT COMPARISON")
print("=" * 60)

# Pipeline
pipe_prompt_embeds, pipe_text_ids = pipe.encode_prompt(
    prompt=prompt,
    device=device,
    max_sequence_length=512,
)
print(f"\n[Pipeline]")
print(f"  prompt_embeds shape: {pipe_prompt_embeds.shape}")
print(f"  prompt_embeds mean: {pipe_prompt_embeds.mean().item():.6f}")
print(f"  text_ids shape: {pipe_text_ids.shape}")

# Trainer
trainer_prompt_embeds, trainer_text_ids = trainer_t2i.encode_prompt(
    prompt=prompt,
    device=device,
    max_sequence_length=512,
)
print(f"\n[Trainer]")
print(f"  prompt_embeds shape: {trainer_prompt_embeds.shape}")
print(f"  prompt_embeds mean: {trainer_prompt_embeds.mean().item():.6f}")
print(f"  text_ids shape: {trainer_text_ids.shape}")

# Compare
print(f"\n[Comparison]")
print(f"  prompt_embeds shape match: {pipe_prompt_embeds.shape == trainer_prompt_embeds.shape}")
if pipe_prompt_embeds.shape == trainer_prompt_embeds.shape:
    diff = (pipe_prompt_embeds - trainer_prompt_embeds.to(pipe_prompt_embeds.dtype)).abs()
    print(f"  prompt_embeds max diff: {diff.max().item():.6f}")

In [ ]:
# ============================================================
# 2. PREPARE LATENTS COMPARISON
# ============================================================
print("=" * 60)
print("2. PREPARE LATENTS COMPARISON")
print("=" * 60)

num_channels_latents = pipe.transformer.config.in_channels // 4

# Pipeline
pipe_latents, pipe_latent_ids = pipe.prepare_latents(
    batch_size=1,
    num_latents_channels=num_channels_latents,
    height=height,
    width=width,
    dtype=pipe_prompt_embeds.dtype,
    device=device,
    generator=torch.Generator(device=device).manual_seed(seed),
)
print(f"\n[Pipeline]")
print(f"  latents shape: {pipe_latents.shape}")
print(f"  latent_ids shape: {pipe_latent_ids.shape}")

# Trainer
trainer_latents, trainer_latent_ids = trainer_t2i.prepare_latents(
    batch_size=1,
    height=height,
    width=width,
    dtype=trainer_prompt_embeds.dtype,
    device=device,
    generator=torch.Generator(device=device).manual_seed(seed),
)
print(f"\n[Trainer]")
print(f"  latents shape: {trainer_latents.shape}")
print(f"  latent_ids shape: {trainer_latent_ids.shape}")

# Compare
print(f"\n[Comparison]")
print(f"  latents shape match: {pipe_latents.shape == trainer_latents.shape}")
print(f"  latent_ids shape match: {pipe_latent_ids.shape == trainer_latent_ids.shape}")

In [ ]:
# ============================================================
# 3. TIMESTEPS COMPARISON
# ============================================================
print("=" * 60)
print("3. TIMESTEPS COMPARISON")
print("=" * 60)

from qflux.models.flux2_loader import compute_empirical_mu
from qflux.utils.sampling import retrieve_timesteps
import copy

image_seq_len = pipe_latents.shape[1]
mu = compute_empirical_mu(image_seq_len, num_inference_steps)
sigmas = np.linspace(1.0, 1 / num_inference_steps, num_inference_steps)

# Fresh schedulers
pipe_scheduler = copy.deepcopy(pipe.scheduler)
trainer_scheduler = copy.deepcopy(trainer_t2i.sampling_scheduler)

# Pipeline timesteps
pipe_scheduler.set_timesteps(sigmas=sigmas, device=device, mu=mu)
pipe_timesteps = pipe_scheduler.timesteps

# Trainer timesteps
trainer_scheduler.set_timesteps(sigmas=sigmas, device=device, mu=mu)
trainer_timesteps = trainer_scheduler.timesteps

print(f"\n[Pipeline]")
print(f"  mu: {mu}")
print(f"  timesteps[:5]: {pipe_timesteps[:5]}")
print(f"  timesteps[-5:]: {pipe_timesteps[-5:]}")

print(f"\n[Trainer]")
print(f"  timesteps[:5]: {trainer_timesteps[:5]}")
print(f"  timesteps[-5:]: {trainer_timesteps[-5:]}")

print(f"\n[Comparison]")
print(f"  timesteps match: {torch.allclose(pipe_timesteps, trainer_timesteps)}")

In [ ]:
# ============================================================
# 4. FULL T2I GENERATION - PIPELINE
# ============================================================
print("=" * 60)
print("4. FULL T2I GENERATION")
print("=" * 60)

print("\n[Generating with Pipeline...]")
with torch.inference_mode():
    pipe_output = pipe(
        prompt=prompt,
        height=height,
        width=width,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=torch.Generator(device=device).manual_seed(seed),
        output_type="pil",
    )
    pipe_image = pipe_output.images[0]

print(f"Pipeline output size: {pipe_image.size}")
from IPython.display import display
display(pipe_image)

In [ ]:
# ============================================================
# 5. FULL T2I GENERATION - TRAINER
# ============================================================
print("\n[Generating with Trainer...]")
trainer_output = trainer_t2i.predict(
    image=None,
    prompt=prompt,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    true_cfg_scale=1.0,
    negative_prompt="",
    weight_dtype=torch.bfloat16,
    height=height,
    width=width,
    output_type='pil',
    generator=torch.Generator(device=device).manual_seed(seed),
)

if isinstance(trainer_output, list):
    trainer_image = trainer_output[0]
else:
    trainer_image = trainer_output

print(f"Trainer output size: {trainer_image.size}")
display(trainer_image)

In [ ]:
# ============================================================
# 6. COMPARISON
# ============================================================
print("=" * 60)
print("6. COMPARISON")
print("=" * 60)

from PIL import Image

# Side-by-side
w1, h1 = pipe_image.size
w2, h2 = trainer_image.size
max_h = max(h1, h2)
max_w = max(w1, w2)

canvas = Image.new('RGB', (max_w * 2 + 20, max_h), color='white')
canvas.paste(pipe_image, (0, 0))
canvas.paste(trainer_image, (max_w + 20, 0))

print("\n[Side-by-Side: Pipeline (left) vs Trainer (right)]")
display(canvas)

# Pixel difference
pipe_arr = np.array(pipe_image.resize((512, 512)))
trainer_arr = np.array(trainer_image.resize((512, 512)))
diff = np.abs(pipe_arr.astype(float) - trainer_arr.astype(float))

print(f"\n[Pixel Difference Statistics]")
print(f"  Max pixel diff: {diff.max():.2f}")
print(f"  Mean pixel diff: {diff.mean():.2f}")
print(f"  Std pixel diff: {diff.std():.2f}")